# Component 1 — Credit readiness: full pipeline

One notebook from data to research result. It replaces reading four notebooks one after another
(kept unchanged in `archive/`): `digital legder.ipynb` (original), `digital_ledger_review.ipynb` (fixes),
`credit_rules_vs_ml.ipynb` (experiments).

**Research question.** Can a small shop's digital ledger (sales, expenses, stock, payments) give a
reliable credit-readiness score — better than the fixed rules banks use — and how much data is needed?

| Part | What it does |
|---|---|
| A | Data: load, how it was generated, the oracle ceiling |
| B | Features and train/test split |
| C | Compare five models with cross-validation, pick the champion, test once |
| D | What drives the score (explainability) |
| E | Loan limit: formula vs model |
| F | Bank rules vs ML vs the live system (+ bootstrap) |
| G | How many shops' records are needed |
| H | Does the app send the same kind of inputs? (training/serving check) |
| I | Save the model |

**Data safety:** `sales & financial.csv` is only **read**. (The original notebook regenerates and
overwrites it in its first cell; this notebook never does.)

In [ ]:
import warnings
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, confusion_matrix, f1_score,
                             mean_absolute_error, precision_score, r2_score, recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
RANDOM_STATE = 42

# Part A — Data

## A1. Load the dataset (read only)

In [ ]:
df = pd.read_csv('sales & financial.csv').drop_duplicates().reset_index(drop=True)
print(f'{len(df)} shops, {df.shape[1]} columns | credit-ready: {df["target_credit_ready"].mean():.0%}')
df = df.drop(columns=['shop_id', 'year_month'], errors='ignore')
df.describe().T[['min', '50%', 'max']].round(2)

## A2. How the data was generated (from `digital legder.ipynb`, cell 1)

The dataset is **synthetic**: each shop's values are drawn at random, and the label comes from a known
score plus random noise:

```
score = 0.05 × profit_margin_pct + 0.09 × months_active + 2.0 × digital_payment_ratio
        − 3.5 × stockout_rate − 1.5 × credit_sales_ratio − 0.15 × sales_volatility
credit_ready = 1  if  sigmoid(score − median score) > a random number between 0 and 1
```

Because the formula is known, the best score any model could reach (the **oracle**) can be computed.
Results below are read against that ceiling. Revenue, expenses, profit and daily transactions are
**not** in the formula.

In [ ]:
def true_score(d):
    return (0.05 * d['profit_margin_pct'] + 0.09 * d['months_active'] + 2.0 * d['digital_payment_ratio']
            - 3.5 * d['stockout_rate'] - 1.5 * d['credit_sales_ratio'] - 0.15 * d['sales_volatility'])

SCORE_MEDIAN = np.median(true_score(df))
def oracle_proba(d):
    return 1 / (1 + np.exp(-(true_score(d) - SCORE_MEDIAN)))

effect = pd.Series({'months_active': 0.09, 'profit_margin_pct': 0.05, 'digital_payment_ratio': 2.0,
                    'stockout_rate': -3.5, 'credit_sales_ratio': -1.5, 'sales_volatility': -0.15})
effect = (effect * df[effect.index].std()).sort_values(key=abs, ascending=False)
print('Effect of each factor in the formula (weight x standard deviation):')
print(effect.round(2).to_string())

# Part B — Features and split

## B1. Domain features (same function as `ml_service/app.py`)

`net_cash_flow`, `debt_to_income_ratio`, `digital_revenue_volume`, `revenue_per_active_month`,
`cash_flow_margin` are computed from the raw columns inside the pipeline, so the app sends only raw values.

In [ ]:
def add_domain_features(X_df):
    X_out = X_df.copy()
    rev = np.maximum(X_out['monthly_revenue_rs'], 1.0) if 'monthly_revenue_rs' in X_out.columns else 1.0
    exp = X_out['monthly_expenses_rs'] if 'monthly_expenses_rs' in X_out.columns else 0.0
    active_m = np.maximum(X_out['months_active'], 1.0) if 'months_active' in X_out.columns else 1.0
    dig_ratio = X_out['digital_payment_ratio'] if 'digital_payment_ratio' in X_out.columns else 0.0
    X_out['net_cash_flow'] = rev - exp
    X_out['debt_to_income_ratio'] = exp / rev
    X_out['digital_revenue_volume'] = rev * dig_ratio
    X_out['revenue_per_active_month'] = rev / active_m
    X_out['cash_flow_margin'] = X_out['net_cash_flow'] / rev
    return X_out

## B2. Stratified 80/20 split and the numeric columns

*Fix from the original:* the original chose numeric columns with `select_dtypes(['int64', 'float64'])`.
The original notebook generates the data in memory with `np.random.randint`, which gives `int32` on
Windows, so `months_active` was silently dropped — the strongest factor in the formula.
`select_dtypes('number')` keeps every numeric type.

In [ ]:
X_raw = df.drop(columns=['target_credit_ready'])
y = df['target_credit_ready'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X_raw, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

num_cols = add_domain_features(X_raw).select_dtypes('number').columns.tolist()
print(f'Train {len(X_train)} / test {len(X_test)} | {len(num_cols)} numeric inputs | months_active included: {"months_active" in num_cols}')
print('dtype of months_active in this file:', df['months_active'].dtype)

def pipeline(estimator):
    pre = ColumnTransformer([('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols)])
    return Pipeline([('feat_eng', FunctionTransformer(add_domain_features, validate=False)), ('prep', pre), ('clf', estimator)])

# Part C — Model comparison

## C1. Five models, 5-fold cross-validation on the training set only

In [ ]:
estimators = {
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    'DecisionTree': DecisionTreeClassifier(max_depth=6, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(n_estimators=300, max_depth=10, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=250, learning_rate=0.05, max_depth=4, subsample=0.85, random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.85, colsample_bytree=0.85,
                             eval_metric='logloss', random_state=RANDOM_STATE),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = {}
for name, est in estimators.items():
    s = cross_validate(pipeline(est), X_train, y_train, cv=cv, scoring=['roc_auc', 'accuracy', 'f1'], n_jobs=-1)
    cv_rows[name] = {'CV ROC-AUC': s['test_roc_auc'].mean(), '± std': s['test_roc_auc'].std(),
                     'CV accuracy': s['test_accuracy'].mean(), 'CV F1': s['test_f1'].mean()}
cv_table = pd.DataFrame(cv_rows).T.round(3)
best_name = cv_table['CV ROC-AUC'].idxmax()
print(f'Champion (chosen on cross-validation only): {best_name}')
cv_table

Logistic regression wins because the true formula is a weighted sum (linear). More complex models
have to approximate a straight line with many splits and do slightly worse.

## C2. Test once, against the oracle ceiling and the saved original model

In [ ]:
champion = pipeline(clone(estimators[best_name])).fit(X_train, y_train)
p_ml = champion.predict_proba(X_test)[:, 1]
p_oracle = oracle_proba(X_test)

rows = {'Champion: ' + best_name: p_ml, 'Oracle (true formula)': p_oracle}
try:
    original = joblib.load('archive/component1_sales_financial_model.pkl')['classifier_pipeline']
    rows = {'Original model (no months_active)': original.predict_proba(X_test)[:, 1], **rows}
except Exception as e:
    print('Original .pkl not loaded:', e)

test_table = pd.DataFrame({k: {'ROC-AUC': roc_auc_score(y_test, p), 'Accuracy': accuracy_score(y_test, p >= .5),
                               'Precision': precision_score(y_test, p >= .5), 'Recall': recall_score(y_test, p >= .5),
                               'F1': f1_score(y_test, p >= .5)} for k, p in rows.items()}).T
ORACLE_AUC = test_table.loc['Oracle (true formula)', 'ROC-AUC']
test_table['% of oracle AUC'] = test_table['ROC-AUC'] / ORACLE_AUC * 100
test_table.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, p_ml >= .5, display_labels=['Not ready', 'Ready'], cmap='Blues', ax=ax)
ax.set_title(f'{best_name} — test set (threshold 0.5)'); plt.tight_layout(); plt.show()

# Part D — What drives the score?

For logistic regression each input's coefficient (on standardised inputs) shows its direction and strength.
The app shows the same idea per shop as "What's affecting this" bars (SHAP).

In [ ]:
coef = pd.Series(champion.named_steps['clf'].coef_[0], index=num_cols).sort_values()
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(coef.index, coef.values, color=['#c0392b' if v < 0 else '#1f5fa8' for v in coef.values])
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Coefficient (standardised input)  -  left lowers the score, right raises it')
ax.set_title('What drives the credit-readiness score'); plt.tight_layout(); plt.show()

The strongest inputs — months active, stock-outs, digital payments and profit margin — are the ones in the
formula; inputs that are not in it (revenue, expenses, daily transactions) get weights near zero. The model
has recovered how the labels were made.

Profit margin appears three times: `profit_margin_pct`, `cash_flow_margin` and `debt_to_income_ratio`
(= 1 − margin) carry the same information, so its weight is split across the three bars. Their combined
effect is about as strong as stock-outs.

# Part E — Loan limit: formula vs model

The original notebook trained a random forest to predict the loan limit. Its target was exactly
`max(0, 3.5 × net cash flow)` — a formula. The formula itself is exact and explainable, so the app uses it.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
eng_train, eng_test = add_domain_features(X_train), add_domain_features(X_test)
limit_train = np.maximum(0, eng_train['net_cash_flow'] * 3.5)[y_train == 1]
limit_test = np.maximum(0, eng_test['net_cash_flow'] * 3.5)[y_test == 1]
reg = Pipeline([('feat_eng', FunctionTransformer(add_domain_features, validate=False)),
                ('prep', ColumnTransformer([('num', StandardScaler(), num_cols)])),
                ('reg', RandomForestRegressor(n_estimators=200, max_depth=8, random_state=RANDOM_STATE, n_jobs=-1))])
reg.fit(X_train[y_train == 1], limit_train)
pred = reg.predict(X_test[y_test == 1])
print(f'RandomForest regressor : R² {r2_score(limit_test, pred):.3f} | MAE LKR {mean_absolute_error(limit_test, pred):,.0f}')
print('The formula itself     : R² 1.000 | MAE LKR 0  -> the app uses the formula')

# Part F — Bank rules vs ML vs the live system

| | Policy | Approves a shop when |
|---|---|---|
| R1 | Bank rules (live hard blocks) | debt-to-income ≤ 85 % and at least 3 months in business |
| R2 | Strict bank rules (original design) | R1 and stock-out rate ≤ 25 % |
| M | ML only | score ≥ 50 |
| H | **Live system** | R1 and score ≥ 50 (`ml_service/app.py`) |
| ★ | Oracle | true formula ≥ 50 % |

**Precision** = good-loan rate (of approved shops, how many are credit-ready) ·
**Recall** = inclusion (of credit-ready shops, how many get an offer).

In [ ]:
y_true = y_test.to_numpy()
rule_basic = ((eng_test['debt_to_income_ratio'] <= 0.85) & (X_test['months_active'] >= 3)).to_numpy()
rule_strict = rule_basic & (X_test['stockout_rate'] <= 0.25).to_numpy()
policies = {
    'R1. Bank rules': rule_basic,
    'R2. Strict bank rules': rule_strict,
    'M. ML only': p_ml >= 0.5,
    'H. Live: rules AND ML': rule_basic & (p_ml >= 0.5),
    '★ Oracle': (p_oracle >= 0.5).to_numpy(),
}
def policy_scores(y, approve):
    tn, fp, fn, tp = confusion_matrix(y, approve, labels=[0, 1]).ravel()
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return {'approval rate %': approve.mean() * 100, 'precision': p, 'recall': r,
            'F1': 2 * p * r / (p + r) if p + r else 0.0, 'ready but rejected': fn, 'approved but not ready': fp}
policy_table = pd.DataFrame({k: policy_scores(y_true, v) for k, v in policies.items()}).T.astype(float).round(3)
policy_table

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
idx = rng.integers(0, len(y_true), size=(1000, len(y_true)))
yb = y_true.astype(bool)[idx]
def f1_boot(f):
    fb = np.asarray(f, dtype=bool)[idx]
    tp = (yb & fb).sum(1); fp = (~yb & fb).sum(1); fn = (yb & ~fb).sum(1)
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)
boot = {k: f1_boot(v) for k, v in policies.items()}
for a, b in [('M. ML only', 'R1. Bank rules'), ('M. ML only', 'R2. Strict bank rules'), ('H. Live: rules AND ML', 'M. ML only')]:
    d = boot[a] - boot[b]; lo, hi = np.percentile(d, [2.5, 97.5])
    print(f'F1({a}) - F1({b}): {d.mean():+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]')

The strict stock-out rule turns away most credit-ready shops — this is why the app shows it as a
**notice**, not a block.

# Part G — How many shops' records are needed?

The model is retrained on random subsets of the training shops (30 times per size) and tested on the
same test shops. (Months of history are not used here: `months_active` is part of the label formula,
so that analysis would only return the formula.)

In [ ]:
sizes = [25, 50, 100, 200, 400, 800, 1200, len(X_train)]
lc = []
for n in sizes:
    aucs = []
    for rep in range(30 if n < len(X_train) else 1):
        if n < len(X_train):
            Xs, _, ys, _ = train_test_split(X_train, y_train, train_size=n, stratify=y_train, random_state=rep)
        else:
            Xs, ys = X_train, y_train
        aucs.append(roc_auc_score(y_test, pipeline(clone(estimators[best_name])).fit(Xs, ys).predict_proba(X_test)[:, 1]))
    aucs = np.array(aucs)
    lc.append({'training shops': n, 'mean AUC': aucs.mean(), 'worst 5% AUC': np.percentile(aucs, 5),
               '% of oracle': aucs.mean() / ORACLE_AUC * 100})
curve = pd.DataFrame(lc).set_index('training shops').round(3)
curve

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
order = ['R1. Bank rules', 'R2. Strict bank rules', 'M. ML only', 'H. Live: rules AND ML']
r = policy_table.loc[order]; x = np.arange(len(order)); w = 0.38
ax[0].bar(x - w / 2, r['precision'], w, label='Precision (good-loan rate)', color='#1f5fa8')
ax[0].bar(x + w / 2, r['recall'], w, label='Recall (inclusion)', color='#e08e79')
ax[0].set_xticks(x); ax[0].set_xticklabels([o.split('. ')[1] for o in order], rotation=12)
ax[0].set_ylim(0, 1.15); ax[0].legend(loc='upper center', ncol=2); ax[0].set_title('Who gets approved (test set)')
ax[1].plot(curve.index, curve['mean AUC'], 'o-', color='#1f5fa8', label='Mean test AUC')
ax[1].plot(curve.index, curve['worst 5% AUC'], 's--', color='#888', label='Worst 5% of runs')
ax[1].axhline(ORACLE_AUC, color='#2e7d32', lw=1, label=f'Oracle ceiling ({ORACLE_AUC:.3f})')
ax[1].set_xscale('log'); ax[1].set_xticks(sizes); ax[1].set_xticklabels(sizes)
ax[1].set_xlabel('Number of training shops (log scale)'); ax[1].set_ylabel('ROC-AUC')
ax[1].legend(loc='lower right'); ax[1].set_title('How many shops are needed to train')
plt.tight_layout(); plt.savefig('credit_readiness_pipeline.png', dpi=120); plt.show()

# Part H — Training/serving check: does the app send the same kind of inputs?

The app computes the inputs from the shop's real ledger (`backend/src/utils/features.js`,
`buildCreditFeatures`). Two inputs are not on the same footing as the training data:

- **`sales_volatility`**: in the training data it is a random number between 0.1 and 5.0 with no
  defined unit. The app computes the coefficient of variation of daily sales (0 to 1.5). The app's values
  sit in the low part of the training range, so a shop always looks a little *steadier* than it would
  in training.
- **`credit_sales_ratio`**: the app has no "sold on credit" payment method, so it sends nothing and the
  model uses the training median (0.25).

How much can this move the score? Below, every test shop's volatility is replaced by the app's range.

In [ ]:
X_app_like = X_test.copy()
rng2 = np.random.default_rng(RANDOM_STATE)
X_app_like['sales_volatility'] = rng2.uniform(0.0, 1.5, len(X_app_like))
p_app_like = champion.predict_proba(X_app_like)[:, 1]
print(f'Average score with training-range volatility: {p_ml.mean() * 100:5.1f}')
print(f'Average score with app-range volatility     : {p_app_like.mean() * 100:5.1f}')
print(f'Shops whose decision (>= 50) changes        : {((p_ml >= .5) != (p_app_like >= .5)).mean():.1%}')

This is reported as a **limitation**: with real lender data, `sales_volatility` should be defined the same
way in training and in the app (coefficient of variation of daily sales), and the model retrained.

# Part I — Save the model

Refit the champion on **all** shops. The app's current model
(`ml_service/models/credit_readiness_model.pkl`) is checked
against it, and the bundle is saved as `credit_readiness_model.pkl`; a copy of it is the app's model.

In [ ]:
final = pipeline(clone(estimators[best_name])).fit(X_raw, y)
bundle = {
    'classifier_pipeline': final,
    'raw_feature_names': X_raw.columns.tolist(),
    'loan_limit_rule': 'max(0, 3.5 x monthly net cash flow)',
    'evaluation': {'champion': best_name,
                   'test_auc': round(float(test_table.loc['Champion: ' + best_name, 'ROC-AUC']), 4),
                   'oracle_auc': round(float(ORACLE_AUC), 4)},
}
try:
    deployed = joblib.load('../../ml_service/models/credit_readiness_model.pkl')['classifier_pipeline']
    diff = np.abs(deployed.predict_proba(X_raw)[:, 1] - final.predict_proba(X_raw)[:, 1]).max()
    print(f'Largest difference from the model the app uses: {diff:.6f}',
          '(same model)' if diff < 1e-4 else '(different)')
except Exception as e:
    print('Deployed model not checked:', e)
joblib.dump(bundle, 'credit_readiness_model.pkl', compress=3)
print('Saved credit_readiness_model.pkl ->', bundle['evaluation'])

# Conclusion

*Written after reading the numbers above — fill in with the actual results:*

- Champion: ___ (test ROC-AUC ___ = ___ % of the oracle ceiling ___).
- ML vs bank rules: F1 ___ vs ___ (interval ___). The strict stock-out rule turns away ___ of ___ credit-ready shops.
- About ___ shops' records are enough to reach ___ % of the ceiling.

**Limitations.** The labels come from a known formula with noise (synthetic data), so the numbers describe
this data, not real loan repayment; the comparisons (rules vs ML, data size) are valid because every method
uses the same labels. `sales_volatility` is defined differently in training and in the app (Part H), and
`credit_sales_ratio` cannot be measured yet. A real study needs repayment outcomes from a lender.